# Enterprise Document Intelligence & Semantic Search — Colab Notebook



In [ ]:
# STEP 1: Install dependencies (Colab has most pre-installed, this ensures the rest)
!pip install -q sentence-transformers faiss-cpu anthropic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 76.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 62.7 MB/s eta 0:00:00


## Step 2: Load the embedding model



In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")
print("Embedding model loaded. Output dimension:", model.get_sentence_embedding_dimension())

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded. Output dimension: 384


/tmp/ipykernel_929/1646566536.py:4: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("Embedding model loaded. Output dimension:", model.get_sentence_embedding_dimension())


## Step 3: Sample documents



In [ ]:
SAMPLE_DOCS = {
    "hr_policy.txt": (
        "Remote Work Policy. Employees may work from home up to three days "
        "per week with manager approval. Home office equipment purchases up "
        "to $500 per year are reimbursable with receipts submitted through "
        "the expense portal within 30 days of purchase. "
        "Parental Leave Policy. Full-time employees are entitled to 16 weeks "
        "of paid parental leave following the birth or adoption of a child."
    ),
    "security_guidelines.txt": (
        "Data Classification Standard. All company data must be classified "
        "as Public, Internal, Confidential, or Restricted. Restricted data "
        "must be encrypted at rest using AES-256. "
        "Incident Response. Any suspected security breach must be reported "
        "to the security team within one hour of discovery via the incident "
        "hotline or the security-incidents Slack channel."
    ),
    "vendor_contract_acme.txt": (
        "Payment Terms. Acme Corp shall invoice monthly in arrears. Payment "
        "is due within Net 45 days of invoice receipt. "
        "Termination Clause. Either party may terminate this agreement with "
        "90 days written notice. Upon termination, Acme Corp shall provide "
        "a full data export within 30 days."
    ),
}

print(f"{len(SAMPLE_DOCS)} sample documents loaded")

3 sample documents loaded


## Step 4: Chunking



In [ ]:
def chunk_text(text, source_name, chunk_size=120, overlap=20):
    words = text.split()
    chunks = []
    start = 0
    while start < len(words):
        end = start + chunk_size
        chunk_words = words[start:end]
        chunks.append({"text": " ".join(chunk_words), "source": source_name})
        if end >= len(words):
            break
        start = end - overlap
    return chunks

all_chunks = []
for name, text in SAMPLE_DOCS.items():
    all_chunks.extend(chunk_text(text, name))

print(f"Produced {len(all_chunks)} chunks from {len(SAMPLE_DOCS)} documents")
for c in all_chunks[:2]:
    print("-", c["source"], ":", c["text"][:80], "...")

Produced 3 chunks from 3 documents
- hr_policy.txt : Remote Work Policy. Employees may work from home up to three days per week with  ...
- security_guidelines.txt : Data Classification Standard. All company data must be classified as Public, Int ...


## Step 5: Embed every chunk



In [ ]:
import numpy as np

texts = [c["text"] for c in all_chunks]
embeddings = model.encode(texts, normalize_embeddings=True, show_progress_bar=True)
embeddings = np.array(embeddings).astype("float32")

print("Embeddings shape:", embeddings.shape)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embeddings shape: (3, 384)


## Step 6: Build the FAISS ANN index


In [ ]:
import faiss

dim = embeddings.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(embeddings)

print(f"FAISS index built with {index.ntotal} vectors of dimension {dim}")

FAISS index built with 3 vectors of dimension 384


## Step 7: Semantic search

Embed the question with the *same* model, then ask FAISS for the closest chunk vectors.

In [ ]:
def search(query, k=3):
    query_vector = model.encode([query], normalize_embeddings=True).astype("float32")
    scores, indices = index.search(query_vector, k)
    results = []
    for score, idx in zip(scores[0], indices[0]):
        if idx == -1:
            continue
        results.append({"score": float(score), **all_chunks[idx]})
    return results

# Try it — note none of these share exact keywords with the source text
demo_queries = [
    "Can I get my home office chair paid for?",
    "How fast do we need to report a data breach?",
    "What happens if we want to end our deal with Acme early?",
]

for q in demo_queries:
    print("QUERY:", q)
    for r in search(q, k=2):
        print(f"   [{r['score']:.3f}] ({r['source']}) {r['text'][:90]}...")
    print()

QUERY: Can I get my home office chair paid for?
   [0.394] (hr_policy.txt) Remote Work Policy. Employees may work from home up to three days per week with manager ap...
   [0.142] (vendor_contract_acme.txt) Payment Terms. Acme Corp shall invoice monthly in arrears. Payment is due within Net 45 da...

QUERY: How fast do we need to report a data breach?
   [0.403] (security_guidelines.txt) Data Classification Standard. All company data must be classified as Public, Internal, Con...
   [0.276] (vendor_contract_acme.txt) Payment Terms. Acme Corp shall invoice monthly in arrears. Payment is due within Net 45 da...

QUERY: What happens if we want to end our deal with Acme early?
   [0.514] (vendor_contract_acme.txt) Payment Terms. Acme Corp shall invoice monthly in arrears. Payment is due within Net 45 da...
   [0.044] (hr_policy.txt) Remote Work Policy. Employees may work from home up to three days per week with manager ap...



## Step 8 (optional): Generate a grounded answer with Claude



In [ ]:
from google.colab import userdata
import anthropic

try:
    api_key = userdata.get("ANTHROPIC_API_KEY")
except Exception:
    api_key = None  # fall back to manual entry below if no Colab secret is set

if not api_key:
    api_key = input("Paste your Anthropic API key (or leave blank to skip): ").strip()

def synthesize_answer(query, results, api_key):
    client = anthropic.Anthropic(api_key=api_key)
    context = "\n\n".join(f"[Source: {r['source']}]\n{r['text']}" for r in results)
    prompt = (
        "Answer the question using ONLY the context below. "
        "Cite the source file for your answer. "
        "If the answer isn't contained in the context, say so plainly.\n\n"
        f"Context:\n{context}\n\nQuestion: {query}"
    )
    message = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=400,
        messages=[{"role": "user", "content": prompt}],
    )
    return message.content[0].text

if api_key:
    query = "Can I get my home office chair paid for?"
    results = search(query, k=3)
    answer = synthesize_answer(query, results, api_key)
    print("QUESTION:", query)
    print("\nANSWER:", answer)
else:
    print("No API key provided — skipping answer generation.")

Paste your Anthropic API key (or leave blank to skip): 
No API key provided — skipping answer generation.


## Step 9 (optional): Launch the full Streamlit UI from inside Colab



In [ ]:
!pip install -q streamlit sentence-transformers faiss-cpu anthropic pyngrok

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
with open('app.py', 'r') as f:
    app_py_content = f.read()
print(app_py_content)

"""
ENTERPRISE DOCUMENT INTELLIGENCE & SEMANTIC SEARCH — STREAMLIT APP
A real, runnable RAG (Retrieval-Augmented Generation) pipeline:

  Upload docs -> Chunk -> Embed (transformer) -> FAISS ANN index
  -> Semantic search on your question -> (optional) LLM answer

Run locally:
    pip install -r requirements.txt
    streamlit run app.py

First run downloads a small embedding model (~80MB) from Hugging Face,
so you need normal internet access the first time you run it.
"""

import numpy as np
import streamlit as st
import faiss
from sentence_transformers import SentenceTransformer

# -----------------------------------------------------------------------
# PAGE CONFIG — must be the first Streamlit command
# -----------------------------------------------------------------------
st.set_page_config(page_title="Document Intelligence Search", layout="wide")


# -----------------------------------------------------------------------
# STAGE 3: EMBEDDING MODEL (the deep learning core of the w

In [ ]:
!pip install -q streamlit
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared-linux-amd64

In [ ]:
import subprocess, time

!pkill -f streamlit
!pkill -f cloudflared

# Start Streamlit in the background
log_file = open("streamlit_log.txt", "w")
subprocess.Popen(
    ["streamlit", "run", "app.py", "--server.port", "8501", "--server.headless", "true"],
    stdout=log_file, stderr=subprocess.STDOUT,
)
time.sleep(10)

# Start the tunnel — no account, no token, works immediately
tunnel_log = open("tunnel_log.txt", "w")
subprocess.Popen(
    ["./cloudflared-linux-amd64", "tunnel", "--url", "http://localhost:8501"],
    stdout=tunnel_log, stderr=subprocess.STDOUT,
)
time.sleep(8)

print(open("tunnel_log.txt").read())

2026-08-06T22:13:26Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-08-06T22:13:26Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-08-06T22:13:31Z INF +--------------------------------------------------------------------------------------------+
2026-08-06T22:13:31Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-08-06T22:13:31Z INF |  https://powerpoint-wales-division-pat.trycloudflare.c

# **FOR ALL TYPES OF FILES**

In [24]:
# ============================================================
# ONE-CELL SETUP: install -> write app.py -> run Streamlit -> tunnel
# ============================================================

!pip install -q streamlit sentence-transformers faiss-cpu anthropic pandas openpyxl
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared-linux-amd64

app_code = '''
import numpy as np
import streamlit as st
import faiss
import pandas as pd
from sentence_transformers import SentenceTransformer

st.set_page_config(page_title="Document Intelligence Search", layout="wide")

@st.cache_resource
def load_embedding_model():
    return SentenceTransformer("all-MiniLM-L6-v2")

def chunk_text(text, source_name, chunk_size=120, overlap=20):
    words = text.split()
    chunks = []
    start = 0
    while start < len(words):
        end = start + chunk_size
        chunks.append({"text": " ".join(words[start:end]), "source": source_name})
        if end >= len(words):
            break
        start = end - overlap
    return chunks

def chunk_tabular(df, source_name, rows_per_chunk=1):
    chunks = []
    for start in range(0, len(df), rows_per_chunk):
        batch = df.iloc[start:start + rows_per_chunk]
        for _, row in batch.iterrows():
            row_text = ", ".join(f"{col}: {val}" for col, val in row.items() if pd.notna(val))
            chunks.append({"text": row_text, "source": source_name})
    return chunks

def build_index(embeddings):
    dim = embeddings.shape[1]
    index = faiss.IndexFlatIP(dim)
    index.add(embeddings.astype("float32"))
    return index

def search(query, model, index, chunks, k=3):
    query_vector = model.encode([query], normalize_embeddings=True).astype("float32")
    scores, indices = index.search(query_vector, k)
    results = []
    for score, idx in zip(scores[0], indices[0]):
        if idx == -1:
            continue
        results.append({"score": float(score), **chunks[idx]})
    return results

def synthesize_answer(query, results, api_key):
    import anthropic
    client = anthropic.Anthropic(api_key=api_key)
    context = "\\n\\n".join(f"[Source: {r['source']}]\\n{r['text']}" for r in results)
    prompt = (
        "Answer the question using ONLY the context below. Cite the source file. "
        "If the answer is not contained in the context, say so plainly.\\n\\n"
        f"Context:\\n{context}\\n\\nQuestion: {query}"
    )
    message = client.messages.create(
        model="claude-sonnet-4-6", max_tokens=400,
        messages=[{"role": "user", "content": prompt}],
    )
    return message.content[0].text

SAMPLE_DOCS = {
    "hr_policy.txt": (
        "Remote Work Policy. Employees may work from home up to three days "
        "per week with manager approval. Home office equipment purchases up "
        "to $500 per year are reimbursable with receipts submitted through "
        "the expense portal within 30 days of purchase."
    ),
    "security_guidelines.txt": (
        "Data Classification Standard. All company data must be classified "
        "as Public, Internal, Confidential, or Restricted. Restricted data "
        "must be encrypted at rest using AES-256. Incident Response. Any "
        "suspected security breach must be reported within one hour."
    ),
    "vendor_contract_acme.txt": (
        "Payment Terms. Acme Corp shall invoice monthly in arrears, due "
        "Net 45 days. Termination Clause. Either party may terminate with "
        "90 days written notice."
    ),
}

st.title("Enterprise Document Intelligence & Semantic Search")
st.caption("Ask questions in plain English. Supports .txt, .csv, and .xlsx files.")

with st.sidebar:
    st.header("Setup")
    api_key = st.text_input("Anthropic API key (optional)", type="password")
    use_sample = st.checkbox("Use built-in sample documents", value=True)
    uploaded_files = st.file_uploader("...or upload files", accept_multiple_files=True, type=["txt", "csv", "xlsx"])
    build_clicked = st.button("Build / rebuild index", type="primary")

if "index" not in st.session_state:
    st.session_state.index = None
    st.session_state.chunks = None

model = load_embedding_model()

if build_clicked:
    all_chunks = []
    if use_sample:
        for name, text in SAMPLE_DOCS.items():
            all_chunks.extend(chunk_text(text, name))
    for f in uploaded_files or []:
        if f.name.endswith(".txt"):
            text = f.read().decode("utf-8")
            all_chunks.extend(chunk_text(text, f.name))
        elif f.name.endswith(".csv"):
            df = pd.read_csv(f)
            all_chunks.extend(chunk_tabular(df, f.name))
        elif f.name.endswith(".xlsx"):
            sheets = pd.read_excel(f, sheet_name=None)
            for sheet_name, df in sheets.items():
                all_chunks.extend(chunk_tabular(df, f"{f.name} [{sheet_name}]"))

    if not all_chunks:
        st.sidebar.error("No documents to index.")
    else:
        with st.spinner(f"Embedding {len(all_chunks)} chunks..."):
            embeddings = model.encode([c["text"] for c in all_chunks], normalize_embeddings=True)
        index = build_index(np.array(embeddings))
        st.session_state.index = index
        st.session_state.chunks = all_chunks
        st.sidebar.success(f"Indexed {len(all_chunks)} chunks")

query = st.text_input("Ask a question about your documents", placeholder="e.g. Which vendor has Net 45 payment terms?")

if query:
    if st.session_state.index is None:
        st.warning("Click Build / rebuild index in the sidebar first.")
    else:
        results = search(query, model, st.session_state.index, st.session_state.chunks, k=3)
        st.subheader("Top matching passages")
        for r in results:
            st.markdown(f"**{r['source']}**  similarity `{r['score']:.3f}`")
            st.write(r["text"])
            st.divider()
        if api_key:
            with st.spinner("Generating grounded answer..."):
                try:
                    st.subheader("AI answer")
                    st.write(synthesize_answer(query, results, api_key))
                except Exception as e:
                    st.error(f"Could not generate an answer: {e}")
        else:
            st.info("Add an Anthropic API key in the sidebar for a synthesized, cited answer.")
'''

with open("app.py", "w") as f:
    f.write(app_code)

print("app.py written successfully.\n")

import subprocess, time
!pkill -f streamlit
!pkill -f cloudflared

log_file = open("streamlit_log.txt", "w")
subprocess.Popen(
    ["streamlit", "run", "app.py", "--server.port", "8501", "--server.headless", "true"],
    stdout=log_file, stderr=subprocess.STDOUT,
)
time.sleep(10)

tunnel_log = open("tunnel_log.txt", "w")
subprocess.Popen(
    ["./cloudflared-linux-amd64", "tunnel", "--url", "http://localhost:8501"],
    stdout=tunnel_log, stderr=subprocess.STDOUT,
)
time.sleep(8)

print("=== TUNNEL OUTPUT (your live URL is in here) ===")
print(open("tunnel_log.txt").read())


app.py written successfully.

=== TUNNEL OUTPUT (your live URL is in here) ===
2026-08-06T22:36:11Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-08-06T22:36:11Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-08-06T22:36:15Z INF +--------------------------------------------------------------------------------------------+
2026-08-06T22:36:15Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
20